### Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

### Read from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12_raw")  
df.display() 

In [0]:
df.printSchema()

### Silver transformations

Rename columns with proper column names.

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birthdate",
    "GEN": "gender",
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

Check distinct gender values.

In [0]:
df.select("gender").distinct().show(truncate=False)

Remove whitespaces from gender values.

In [0]:
df = df.withColumn(
    "gender",
    trim(col("gender"))
)

Replace "M" with "Male" and "F" with "Female".

In [0]:
df = df.replace(
    "M",
    "Male",
    subset=["gender"]
)

df = df.replace(
    "F",
    "Female",
    subset=["gender"]
)

df.groupBy("gender").count().show()

In [0]:
df.display()

### Write into Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

In [0]:
%sql
SELECT * 
FROM workspace.silver.erp_customers 
LIMIT 10